# ENT Triage Model Finetuning

Finetune a small LLM (Qwen2) for ENT triage using synthetic transcript→output pairs.

**Output format:** SUMMARY, FINDINGS, FLAGS, URGENCY (routine, semi-urgent, or urgent), REASONING

**Deployment note:** Groq does *not* host custom finetuned models. After finetuning, deploy to **Together.ai**, **Replicate**, or **Ollama** (local).

In [ ]:
# Install dependencies (run once)
!pip install -q transformers peft datasets accelerate bitsandbytes

In [ ]:
import json
from pathlib import Path
from datasets import Dataset
import torch

## 1. Load or upload training data

**Recommended (3+ sentence summaries):** Use `combined_triage_training.jsonl` (run `python modelling/code/prepare_training_data.py` first) or `triage_training_data_min3sentences.jsonl`.

- **Option A:** Upload your chosen file to Colab (e.g. `combined_triage_training.jsonl`).
- **Option B:** Use sample inline for a quick test.

In [ ]:
# Option A: Load from uploaded file
# !pip install -q -U google-colab
# from google.colab import files
# uploaded = files.upload()  # Choose triage_training_data.jsonl
# DATA_PATH = list(uploaded.keys())[0]

# Option B: Use sample inline
SAMPLE_DATA = [
    {
        "transcript": "chief_complaint:mild sore throat. symptom_duration:3 days. symptom_severity:mild. symptom_progression:improving. red_flags:No. risk_factors:No",
        "output": "SUMMARY: Patient presents with mild sore throat of 3 days duration. Symptoms improving. No red flags.\nFINDINGS:\n- Mild sore throat\n- 3-day duration\n- Improving trend\nFLAGS: [SYMPTOM] sore throat, [SEVERITY] mild, [DURATION] 3 days, [PROGRESSION] improving\nURGENCY: routine\nREASONING: Mild symptoms, improving, no red flags. Routine appointment."
    },
    {
        "transcript": "chief_complaint:difficulty breathing. symptom_duration:few hours. symptom_severity:severe. symptom_progression:rapidly worsening. red_flags:Yes. risk_factors:No",
        "output": "SUMMARY: Patient with acute difficulty breathing, rapidly worsening. Critical red flag.\nFINDINGS:\n- Difficulty breathing\n- Rapid deterioration\nFLAGS: [RED_FLAG] difficulty breathing, [SEVERITY] severe\nURGENCY: urgent\nREASONING: Difficulty breathing is a critical red flag. Same-day evaluation required."
    },
]

def load_jsonl(path: str):
    data = []
    with open(path) as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

# Prefer combined (3+ sentence) data; fallback to min3sentences, training_data (instruction+input+output), or original
for candidate in ('combined_triage_training.jsonl', 'triage_training_data_min3sentences.jsonl', 'training_data.jsonl', 'triage_training_data.jsonl'):
    try:
        DATA_PATH = candidate
        data = load_jsonl(DATA_PATH)
        print(f"Loaded {len(data)} examples from {DATA_PATH}")
        break
    except FileNotFoundError:
        continue
else:
    data = SAMPLE_DATA
    print(f"Using {len(data)} sample examples (upload combined_triage_training.jsonl for full training)")

# Limit for quick run; set to None to use full dataset
MAX_EXAMPLES = None
if MAX_EXAMPLES is not None:
    data = data[:MAX_EXAMPLES]

# Oversample semi-urgent so model sees more examples per epoch
import re
def _parse_urgency(text):
    m = re.search(r'URGENCY:\s*(\w+(?:-\w+)?)', text, re.IGNORECASE)
    return m.group(1).lower() if m else None
semi_examples = [r for r in data if _parse_urgency(r.get("output", "")) == "semi-urgent"]
if semi_examples:
    data = data + semi_examples  # Repeat semi-urgent 2x total
    print(f"Oversampled {len(semi_examples)} semi-urgent examples (now {len(data)} total)")

In [ ]:
# Convert to chat format for instruction tuning
# Handles: (instruction, input, output) e.g. training_data.jsonl | (transcript, output) e.g. combined_triage_training
def to_chat(r):
    if "instruction" in r and "input" in r:
        user_content = f"{r['instruction']}\n\n{r['input']}"
    else:
        user_content = f"""You are an ENT triage expert. Analyze this transcript and produce a structured triage output.
Include: (1) SUMMARY: 3+ sentence clinical summary, (2) FINDINGS: bullet points, (3) FLAGS: tagged items, (4) URGENCY: routine, semi-urgent, or urgent, (5) REASONING: brief justification. Prioritize red flags.

URGENCY must be exactly one of: routine, semi-urgent, or urgent. Use semi-urgent when: moderate severity + worsening, or fever + non-critical symptoms, or ear discharge, or symptoms needing evaluation within 24–48 hours.

TRANSCRIPT:
{r['transcript']}"""
    return {
        "messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": r["output"]},
        ]
    }

chat_data = [to_chat(r) for r in data]
dataset = Dataset.from_list(chat_data)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
print(dataset)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2-0.5B-Instruct"  # Small for Colab; use Qwen2-1.5B-Instruct for better quality

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments, Trainer

# Ensure pad token is set (Qwen2 often uses EOS as pad)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

def tokenize(examples):
    out = tokenizer(
        examples["text"],
        truncation=True,
        max_length=1024,
        padding="max_length",
        return_tensors=None,
    )
    # Causal LM needs labels; use input_ids and mask padding with -100 so loss is not computed on padding
    out["labels"] = [
        [tid if tid != tokenizer.pad_token_id else -100 for tid in ids]
        for ids in out["input_ids"]
    ]
    return out

train_ds = dataset["train"].map(format_chat, remove_columns=dataset["train"].column_names)
train_ds = train_ds.map(tokenize, batched=True, remove_columns=["text"])
train_ds = train_ds.with_format("torch")

training_args = TrainingArguments(
    output_dir="./triage_qwen_lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
)
trainer.train()

In [ ]:
# Save LoRA adapter
trainer.save_model("./triage_qwen_lora")
tokenizer.save_pretrained("./triage_qwen_lora")

# Optional: Merge LoRA into base model and export for Ollama/Together
# merged = model.merge_and_unload()
# merged.save_pretrained("./triage_qwen_merged")

print("Saved to ./triage_qwen_lora. Download and deploy to Together.ai, Replicate, or Ollama.")

## Evaluate finetuned model

In [ ]:
import re

def parse_urgency(text):
    m = re.search(r'URGENCY:\s*(\w+(?:-\w+)?)', text, re.IGNORECASE)
    return m.group(1).lower() if m else None

# Evaluate on test set (sample for speed; set EVAL_N to None for full test)
EVAL_N = 20
test_data = dataset["test"]
if EVAL_N is not None:
    test_data = test_data.select(range(min(EVAL_N, len(test_data))))

model.eval()
correct, total = 0, 0
results = []

for i, ex in enumerate(test_data):
    user_only = [{"role": "user", "content": ex["messages"][0]["content"]}]
    prompt = tokenizer.apply_chat_template(
        user_only, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    pred_text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    expected = ex["messages"][1]["content"]
    pred_urg = parse_urgency(pred_text)
    exp_urg = parse_urgency(expected)
    if pred_urg and exp_urg:
        correct += pred_urg == exp_urg
        total += 1
    results.append((ex["messages"][0]["content"][:80] + "...", expected, pred_text))

if total > 0:
    print(f"URGENCY accuracy: {correct}/{total} = {100*correct/total:.1f}%")
print("\n--- Sample comparisons (User truncated | Expected | Predicted) ---")
for i, (user_preview, exp, pred) in enumerate(results[:5]):
    print(f"\n--- Example {i+1} ---")
    print(f"User: {user_preview}")
    print(f"Expected:\n{exp[:300]}...")
    print(f"Predicted:\n{pred[:300]}...")

## Deploy Options (Groq does NOT host custom models)

1. **Together.ai** – Upload merged model or LoRA; use their inference API.
2. **Replicate** – Package model as a Replicate model; call via their API.
3. **Ollama** – Convert to GGUF, run locally: `ollama create triage -f Modelfile`.